# Source-disjoint seed worker

One seed per runtime, so the three seeds run in parallel. Set `SEED` below. The partition manifest was written by `colab_D` to Drive (`polyglotfake/source_disjoint/split_manifest.json`); this notebook copies only the clips that manifest names (about 6.5 GB, 25 min) and resumes any stage the seed's directory already holds. Use an A100 or L4 runtime: the XceptionNet stage is bound by CPU augmentation and a T4 runtime's two cores give 31 min per epoch.

In [ ]:
SEED = 43  # 42, 43 or 44

In [ ]:
!nvidia-smi -L; nproc
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b revision/round-3 https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo
%cd /content/repo
!pip -q install speechbrain timm librosa opencv-python-headless facenet-pytorch
!git log --oneline -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROCESSED = '/content/drive/MyDrive/polyglotfake/processed'
OUT = '/content/drive/MyDrive/polyglotfake/source_disjoint'
assert os.path.exists(f'{OUT}/split_manifest.json'), 'run colab_D first: the manifest is missing'
print(open(f'{OUT}/split_manifest.json').read()[:400])

## Stage only the clips the partition uses

In [ ]:
!python -u tools/source_disjoint/stage_split.py --manifest "$OUT/split_manifest.json" --processed "$PROCESSED" --dst /content/pgf_sd
!du -sh /content/pgf_sd/*

## Train, fine-tune and score this seed

Resumable: DONE markers in `$OUT/seed<SEED>` skip finished stages; the XceptionNet script resumes from its own best checkpoint.

In [ ]:
import os
os.environ['SEED_STR'] = str(SEED)
!python -u tools/source_disjoint/train_all.py --data_dir /content/pgf_sd --out_dir "$OUT/seed$SEED_STR" --seed $SEED_STR --finetune --score 2>&1 | tail -n 30

## Read-out over whichever seeds have finished

In [ ]:
import glob
runs = sorted(d for d in glob.glob(f'{OUT}/seed*') if os.path.exists(f'{d}/scores_test.csv.DONE'))
print('finished runs:', runs)
if runs:
    os.environ['RUNS'] = ' '.join(f'"{r}"' for r in runs)
    !python -u tools/source_disjoint/analyse.py --runs $RUNS --out "$OUT/readout.json"